# Feature Engineering Basics

A model can only learn from what its features show it. This notebook covers the core techniques that turn raw data into features a model can use: filling in missing values, encoding categories, binning, scaling and creating new features, all with scikit-learn tools that **learn** from the training data and apply the same rules to new data.

## Learning Objectives

At the end of this notebook, you should be able to:

- Impute missing values with `SimpleImputer`, fitted on the training data only.
- Encode nominal features with `OneHotEncoder` and ordinal features with `OrdinalEncoder`.
- Discretise a numeric feature into bins with `KBinsDiscretizer`.
- Apply min-max scaling and a log transformation to the training and the test data.
- Create a new feature from existing columns, such as a ratio, and compare its correlation with the target.

## Introduction

What is our __goal__ when we __train a machine learning model__ (ML)?

<img src = "./assets/basic_model.png" width=400>

The primary goal when training a machine learning model is to build a system that makes accurate predictions or decisions based on input data. But how do we improve the accuracy of these predictions?

**Selecting the right model** is essential, but it is only the first step. Beyond model selection, we need other strategies that improve what the model learns, so that it makes more **accurate predictions**.

Consider how the model learns: given the `X_train` input, the algorithm finds the parameters that bring the model's output as close as possible to the known `y_train` values.

To improve the model beyond a basic level, we must actively support this learning process. Which strategies can we use to do this?


## What is Feature Engineering?
The diagram below is **oversimplified**. Feeding the algorithm all the raw data produces a model that is just as raw and noisy as its input. Instead, we need to **clean** the data and make **informed decisions** to select the **key features** that allow the model to predict outcomes accurately.

<img src = "./assets/feature_eng.png" width=500>

Some aspects of **feature engineering** are methodical and consistent. We explore these first.

Other aspects depend more on **judgement**: they require a deep understanding of the subject and some **domain intuition**.

**Feature engineering** can be one of the most **time-consuming** parts of modelling, but it is essential. Without it, the model struggles to detect patterns in noisy data.


**Estimated Time Spent on Data Preparation**

<img src = "./assets/stacked-chart.jpeg" width=400>

### Feature Engineering Techniques

| Technique | Purpose |
|:--------------------:|:--------------------------------------------------------------------------------:|
| Imputation | fills in missing values in the data |
| Discretisation | groups the values of a numerical feature into bins |
| Categorical Encoding | encodes categorical features as numerical values |
| Feature Splitting | splits a feature into several parts |
| Feature Scaling | handles the sensitivity of ML algorithms to the scale of input values |
| Feature Expansion | derives new features from existing ones |
| Log Transformation | handles skewed or heteroscedastic data |
| Outlier Handling | handles unusually high or low values in the dataset |
| RBF Transformation | encodes a numeric feature by its similarity to chosen reference values |

**Feature splitting** needs a column that holds several pieces of information, such as a car name (brand and model) or a timestamp (date, weekday and hour). The penguin data has no such column, so this notebook does not demonstrate it. Section 5.3 shows the closely related idea of **creating a feature by hand** from existing columns.

### Feature Engineering Best Practices

#### 1. Split the Dataset into Train and Test Sets as Early as Possible

The order of steps is flexible: for example, you could remove NaNs from the entire dataset before splitting. As a good machine learning habit, however, handle missing values **after splitting**. If you remove or impute missing values before splitting, information from the test set can influence the training process, which leads to overly optimistic performance estimates.

#### 2. Engineer the Features of the Training Data

Feature engineering includes any preprocessing technique, for example:

- Dropping missing values
- Converting strings or other non-numeric values into numeric values
- Combining features
- Creating new features

#### 3. Process the Test Data in the Same Way as the Training Data

Apply exactly the same transformations to the test data that you applied to the training data.


---

## Example: Penguin Data

### Load Packages

Documentation: [`set_config`](https://scikit-learn.org/stable/modules/generated/sklearn.set_config.html).

In [ ]:
# data analysis stack
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn import set_config
from sklearn.model_selection import train_test_split

sns.set_style("whitegrid")

# Make every scikit-learn transformer return a pandas DataFrame instead of a NumPy array
set_config(transform_output="pandas")

### Load Data

In [ ]:
df = pd.read_csv("./data/penguins.csv")
df.head()

### Quick Exploration

In [ ]:
df.info()

In [ ]:
df.describe()

### Features and Target

In [ ]:
numerical_features = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm"]

categorical_features = ["species", "island", "sex"]

features = numerical_features + categorical_features

target_variable = "body_mass_g"

### Feature-Target Separation

In [ ]:
# Feature matrix
X = df[features]

# Target column
y = df[target_variable]

### Train-Test Split

Documentation: [`train_test_split`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, shuffle=True
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

This code **splits** the dataset into a training set and a test set, with 25% of the data reserved for testing.

- `random_state` makes the **split** reproducible: the same split occurs each time the code runs.
- `shuffle=True` shuffles the data before splitting.


### Exploratory Data Analysis

For **Exploratory Data Analysis (EDA)**, concatenate `X_train` and `y_train`. This also lets you analyse the relationship between the features and the target variable.

In [ ]:
# Assuming X_train is a DataFrame and y_train is a Series
df_train = pd.concat([X_train, y_train], axis=1)

print("Combined train data shape:", df_train.shape)

**Show Some Plots**

In [ ]:
sns.pairplot(df_train, corner=True, hue="island");

The **pairplot** above shows both the distribution of single variables and the relationships between pairs of variables.

1. **Scatter plots**: each plot below the diagonal shows how two variables relate to each other. For example, in the plot where `flipper_length_mm` meets `bill_length_mm`, each point represents these two measurements for one penguin.

2. **Histograms and density plots**: the plots on the diagonal show how often different values occur for a single variable. For instance, the plot for `bill_depth_mm` shows the distribution of bill depths across all penguins.

3. **Colour**: the colours represent the categories of `island`. This shows at a glance whether the measurements differ by island.


You can also create pairplots that use `sex` or `species` as the `hue` variable.

In [ ]:
df_train

> **Note**: the **indices** in the DataFrame above appear in **random order** because the data was **shuffled** before the split.


### Issues with the Data

**Many models cannot handle** missing values, **categorical features** with non-numeric values, or **numerical features** with very different magnitudes. Data preprocessing prepares the data for these models.


**Check Missing Values**

First, let's take a look at what's missing:

In [ ]:
df_train.isna().sum()

`isna()` and `isnull()` are identical methods: both return a boolean mask where `True` marks a missing value. You can use this mask to filter the rows that contain missing values.

In [ ]:
# To find the rows with NaN
df_train.loc[df_train.isna().any(axis=1)]

In [ ]:
# check missing values graphically
plt.figure(figsize=(7, 5), dpi=100)
sns.heatmap(df_train.isna());

Both views show two missing values in each of the `bill_depth_mm` and `flipper_length_mm` columns, and six in the `sex` column. How do we deal with them?

---

## 1. Imputation: Filling in the Blanks

### What Can We Do with Missing Information?
There are a few strategies:

- __Drop__:
    + rows with missing values
    + columns with many missing values
- __Fill with a value__:
    + __mean__/__median__/__mode__ of a column
    + __interpolate__ / __back fill__ / __forward fill__
    + __mean__/__median__/__mode__ of a group

With `pandas`:

- `df.isna()`: checks for NaNs; combine it with a sum or a heatmap
- `df.dropna()`: drops NaNs
- `df.fillna()`: fills NaNs

Pass `inplace=True` to these methods to modify the DataFrame directly.

---

### 1.1 `SimpleImputer`

We can use the scikit-learn [`SimpleImputer()`](https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html) to impute the missing data in a column. The following strategies are available:

* `strategy='mean'`: **the default option**, numeric data only
* `strategy='median'`: numeric data only
* `strategy='most_frequent'`: the mode, numeric or categorical data
* `strategy='constant'`: requires the additional argument `fill_value`, numeric or categorical data

If your missing values are not NaN (for example `None`, 0, 999 or `"badvalue"`), use the `missing_values` argument to tell the imputer what to look for.

In [ ]:
from sklearn.impute import SimpleImputer

**Impute the `sex` Column with the Most Frequent Value**


In [ ]:
X_train["sex"].value_counts(dropna=False)

In [ ]:
# Instantiating a SimpleImputer object
sex_imputer = SimpleImputer(strategy="most_frequent")

In [ ]:
# Fit the variable imputer on the 'sex' column training data
sex_imputer.fit(X_train[["sex"]])

`.fit()` teaches the imputer what to insert. Here the imputer scans the `sex` column of the training data (`X_train`) to find its most frequent value. After fitting, the imputer stores this mode in its `statistics_` attribute.

In [ ]:
sex_imputer.statistics_

In [ ]:
# Applying the transformation to the 'sex' column of the training data using the pre-fitted imputer
sex_imputed_df_train = sex_imputer.transform(X=X_train[["sex"]])
sex_imputed_df_train

The `transform` method uses the fitted imputer to fill in the missing values in the `sex` column. Because we set `transform_output="pandas"` at the top of the notebook, it returns a DataFrame with the same index and column name as the input, so the result lines up with the rest of `X_train`.

In [ ]:
sex_imputed_df_train.value_counts(dropna=False)

**How Should We Impute Missing Values in the Test Data?**

When you impute missing values, or apply any other transformation, to the **test data**, **do not use any information from outside the training data**. This prevents **data leakage** during model building. Apply only transformations whose parameters were learned from the training data: in scikit-learn, this means calling only the __transform method__ of the imputer or other transformer on the test data.

> [!IMPORTANT]
> As shown in the [machine learning workflow](machine_learning_workflow.md#the-machine-learning-workflow), it is good practice to unlock and transform the test data only at the very end.

In [ ]:
sex_imputed_df_test = sex_imputer.transform(X_test[["sex"]])
sex_imputed_df_test

---

**Impute the `flipper_length_mm` and `bill_depth_mm` Columns with the Median**

We follow the same steps as for the `sex` column to fill the missing values in the `flipper_length_mm` and `bill_depth_mm` columns. This time we use the **median** strategy.

In [ ]:
# Instantiating a SimpleImputer object
flipper_bill_imputer = SimpleImputer(strategy="median")
flipper_bill_imputer

In [ ]:
# Fit the variable imputer on the flipper_length_mm' and 'bill_depth_mm' columns of the training data
flipper_bill_imputer.fit(X_train[["flipper_length_mm", "bill_depth_mm"]])

In [ ]:
# Check the median parameters stored in the imputer after the fit
flipper_bill_imputer.statistics_

In [ ]:
X_train[["flipper_length_mm", "bill_depth_mm"]].median()

In [ ]:
# Applying the transformation to the flipper_length_mm' and 'bill_depth_mm' columns of the training data using the pre-fitted imputer.
flipper_bill_imputed_df_train = flipper_bill_imputer.transform(
    X_train[["flipper_length_mm", "bill_depth_mm"]]
)
flipper_bill_imputed_df_train.isna()

---

## 2. Categorical Encoding: Replacing Categories with Numbers

Most algorithms cannot handle strings, but scikit-learn provides tools to convert categories into numbers.


In W3D1 (notebook 05 of the linear regression repository) you created **dummy variables** with `pd.get_dummies()`, and dropped one column to avoid the dummy variable trap. `OneHotEncoder` does the same job the scikit-learn way, with one important difference: it **learns** the categories on the training data, and then creates exactly the same columns for any new data, even if a category is missing there or a new one appears. That is what you need once preprocessing runs inside a pipeline (notebook 03).

### 2.1 `OneHotEncoder` for Nominal Variables (Categories Without Inherent Order)
We can use the scikit-learn [`OneHotEncoder()`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) to create dummy variables for categorical data. It converts each category of a column into a separate binary column: a row has **1** in the column of its category and **0** in the others. To avoid perfect collinearity (a common statistical issue for some models), there are two options for choosing which columns to drop:

* `drop='first'`: drops the first category of each feature. If a feature has only one category, the whole feature is dropped.
* `drop='if_binary'`: drops a category only if the feature is binary (for example yes/no or on/off). Features with one category or more than two categories remain unchanged.

After transforming the data, use `get_feature_names_out()` to get the names of the new columns.


In [ ]:
from sklearn.preprocessing import OneHotEncoder

What are the **unique categories** in the `species` column?

In [ ]:
X_train["species"].unique()

In [ ]:
# Instantiating a OneHotEncoder object
species_ohencoder = OneHotEncoder(
    drop="first", handle_unknown="ignore", sparse_output=False
)

+ The `handle_unknown` parameter tells the encoder how to deal with categories in new data that were not present when it was fitted: `"ignore"` encodes them as all zeros instead of raising an error.
+ By default, `OneHotEncoder` returns a **sparse matrix**, which stores only the non-zero values and saves memory when there are many categories. A DataFrame needs every value, so we set `sparse_output=False`.

In [ ]:
# Fit the variable encoder on the 'species' column of the training data
species_ohencoder.fit(X=X_train[["species"]])

In [ ]:
# unique categories of the train species column learnt and stred in the encoder
species_ohencoder.categories_

In [ ]:
# new columns/featureeaure of the categories
species_ohencoder.get_feature_names_out()

**Note** that `species_Adelie` has been dropped: it is the reference category.

In [ ]:
species_encoded_train_df = species_ohencoder.transform(X=X_train[["species"]])
species_encoded_train_df

In [ ]:
# Let's see the encoded species columns for two penguins
species_encoded_train_df.loc[[24, 323]]

**How to Read the DataFrame Above**

+ Penguin 24 belongs to neither `Chinstrap` nor `Gentoo`, so it is an `Adelie` penguin (the reference category).
+ Penguin 323 belongs to the species `Gentoo`.

### Try It: Encode `island`

One-hot encode the `island` column of `X_train` in the same way. How many columns do you get, and which island becomes the dropped reference category?

In [ ]:
# TODO: create a OneHotEncoder with drop="first", handle_unknown="ignore" and sparse_output=False
island_encoder = ...

# TODO: fit it on the island column of X_train, then transform that column
island_encoded_df_train = ...
island_encoded_df_train.head()

<details><summary>Show answer</summary>

```python
island_encoder = OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False)
island_encoded_df_train = island_encoder.fit_transform(X_train[["island"]])
island_encoded_df_train.head()
```

There are three islands, so the encoder creates two columns, `island_Dream` and `island_Torgersen`. `Biscoe` comes first in alphabetical order, so it is dropped and becomes the reference category: a penguin from Biscoe has 0 in both columns.

</details>

### 2.2 `OrdinalEncoder` for Ordinal Variables (Categories with Inherent Order)


We can use the scikit-learn [`OrdinalEncoder()`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OrdinalEncoder.html) to encode ordinal categories as numbers. It is useful for categories with a natural order. The key parameters are:

* `categories`: sets the order of the categories explicitly, if the default alphabetical order is not the one you want.
* `dtype`: the data type of the output (default `float64`).
* `handle_unknown`: decides what happens to unknown categories during transformation:
  * `handle_unknown='error'`: **the default option**, raises an error when an unknown category appears.
  * `handle_unknown='use_encoded_value'`: assigns the integer given in the additional argument `unknown_value` to unknown categories.

If your categories have a meaningful order, pass it to the `categories` argument so that the encoding respects it.


The penguins dataset has no ordinal feature, so let's consider a simple example: "Education Level". Its categories, in order of educational achievement, are:

+ High School
+ Bachelor's
+ Master's
+ Doctorate

Ordinal encoding assigns each category a unique integer based on its position in this order:

+ High School --> 0
+ Bachelor's --> 1
+ Master's --> 2
+ Doctorate --> 3


In [ ]:
from sklearn.preprocessing import OrdinalEncoder

In [ ]:
train_data = [
    ["High School"],
    ["Bachelor's"],
    ["Doctorate"],
    ["Master's"],
    ["Doctorate"],
]
df_train_data_ = pd.DataFrame(data=train_data, columns=pd.Index(["education_level"]))
df_train_data_

In [ ]:
# Instantiate the encoder object
edu_encoder = OrdinalEncoder(
    categories=[["High School", "Bachelor's", "Master's", "Doctorate"]], dtype=int
)

In [ ]:
# Fit the variable encoder on the 'education_level' column of the training data
edu_encoder.fit(df_train_data_[["education_level"]])

In [ ]:
# learned and stored categories by the encoder
edu_encoder.categories_

In [ ]:
# Encode the 'education_level' column in the train data
edu_encoded_df_train = edu_encoder.transform(df_train_data_)
edu_encoded_df_train

### Encoding at a Glance

- `OrdinalEncoder()` and `OneHotEncoder()` turn categorical data into integers.
- `OrdinalEncoder()` produces a single column.
- `OneHotEncoder()` produces multiple columns.

Dropping one dummy column (`drop='first'` in `OneHotEncoder`, `drop_first=True` in `pd.get_dummies()`) avoids **perfect collinearity**, which matters especially for regression models. There is some discussion of why multicollinearity is [a problem](https://towardsdatascience.com/multicollinearity-why-is-it-bad-5335030651bf) and [why it might not be that bad](https://stats.stackexchange.com/questions/224051/one-hot-vs-dummy-encoding-in-scikit-learn). As a general rule, avoid it when you use regression models.

- `OrdinalEncoder` is similar to [factorize()](https://pandas.pydata.org/docs/reference/api/pandas.factorize.html) in pandas: both replace each category with an integer.
- `OneHotEncoder` is similar to [get_dummies()](https://pandas.pydata.org/docs/reference/api/pandas.get_dummies.html) in pandas.

---

## 3. Discretisation (Binning): Splitting Numbers into Categories
Breaking a continuous variable into bins has several effects:
+ it can reduce the model's sensitivity to small fluctuations and noise in the data, which reduces the risk of overfitting
+ it can lose important details in the data if the bin widths are poorly chosen, which increases the risk of underfitting
+ it lets linear models capture non-linear relationships:
  + [regression example from scikit-learn](https://scikit-learn.org/stable/auto_examples/preprocessing/plot_discretization.html)
  + [classification example from scikit-learn](https://scikit-learn.org/stable/auto_examples/preprocessing/plot_discretization_classification.html#sphx-glr-auto-examples-preprocessing-plot-discretization-classification-py)

### `KBinsDiscretizer`

We can use [`KBinsDiscretizer()`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.KBinsDiscretizer.html) to sort numerical values into bins and one-hot encode these bins in a single step. The key parameters are:

* `n_bins`: the number of bins to create, default = 5.
* `strategy='quantile'`: **the default option**, creates bins with the same number of values.
* `strategy='uniform'`: creates bins of equal width.
* `encode='onehot'`: **the default option**, one-hot encodes the result and returns a sparse matrix. The code below uses `encode='onehot-dense'`, which returns a dense result that can be shown as a DataFrame.

In [ ]:
sns.displot(data=X_train, x="bill_length_mm", bins=5);

In [ ]:
from sklearn.preprocessing import KBinsDiscretizer

In [ ]:
# Instantiate  KBinsDiscretizer object
bill_length_binner = KBinsDiscretizer(
    n_bins=5, encode="onehot-dense", strategy="quantile"
)

In [ ]:
# Fit the variable binner on the 'bill_length_mm' column of the training data
bill_length_binner.fit(X_train[["bill_length_mm"]])

In [ ]:
# Access and print the bin edges for the 'bill_length_mm' feature
bin_edges = bill_length_binner.bin_edges_
print("Bin edges for 'bill_length_mm':", bin_edges[0])

The bin edges are the **quintiles**: they divide the data into 5 roughly equal parts.

In [ ]:
# Get the quintiles
X_train[["bill_length_mm"]].quantile(q=[0.0, 0.2, 0.4, 0.6, 0.8, 1.0]).rename(
    {"bill_length_mm": "bill_quintiles"}, axis=1
)

In [ ]:
# Plotting the bill length distribution using the learnt bin edges
sns.displot(data=X_train, x="bill_length_mm", bins=bin_edges[0]);

As the plot above shows, the bins have roughly equal heights.

In [ ]:
# Transform 'bill_length_mm' using the quintile bins defined in 'bill_length_binner'
bill_length_binned_df_train = bill_length_binner.transform(X_train[["bill_length_mm"]])
bill_length_binned_df_train

The following table shows the mapping between the one-hot-encoded categories and the bill length ranges:

| Category | Bill Length Range (mm) |
|----------|----------------|
| 0        | 32.1 - 38.6   |
| 1        | 38.6 - 42.3|
| 2        | 42.3 - 46.2|
| 3        | 46.2 - 49.5|
| 4        | 49.5 - 59.6|


**How to Read the Transformed DataFrame Above**
+ The transformed DataFrame assigns each penguin's `bill_length_mm` to one of five quintile categories (0 to 4). Each category corresponds to a range of bill lengths.
+ Penguin 24 is in category 0, so its bill length lies between the minimum 32.1 and the first quintile 38.6.


In [ ]:
# Bill_length value for the observation 24
X_train[["bill_length_mm"]].loc[24]

- `KBinsDiscretizer(strategy='quantile')` is similar to [qcut()](https://pandas.pydata.org/docs/reference/api/pandas.qcut.html) in pandas.
- `KBinsDiscretizer(strategy='uniform')` is similar to [cut()](https://pandas.pydata.org/docs/reference/api/pandas.cut.html) in pandas.

### Try It: Equal-Width Bins

Bin `flipper_length_mm` into 4 bins of **equal width** (`strategy="uniform"`) instead of equal size. Print the bin edges and count how many penguins fall into each bin. Why are the counts no longer equal?

In [ ]:
# TODO: create a KBinsDiscretizer with 4 bins, encode="ordinal" and strategy="uniform"
flipper_binner = ...

# TODO: fit it on the flipper_length_mm column of X_train (drop the missing values first)
flipper = X_train[["flipper_length_mm"]].dropna()
...

print("Bin edges:", flipper_binner.bin_edges_[0])
flipper_binner.transform(flipper).value_counts().sort_index()

<details><summary>Show answer</summary>

```python
flipper_binner = KBinsDiscretizer(n_bins=4, encode="ordinal", strategy="uniform")
flipper = X_train[["flipper_length_mm"]].dropna()
flipper_binner.fit(flipper)

print("Bin edges:", flipper_binner.bin_edges_[0])
flipper_binner.transform(flipper).value_counts().sort_index()
```

The bin edges are 172, 186.5, 201, 215.5 and 230 mm: every bin is 14.5 mm wide. The counts are 32, 108, 64 and 50 penguins. They are no longer equal because penguins are not spread evenly over the range: most Adelie and Chinstrap penguins have flippers between 186.5 and 201 mm, while the larger Gentoo penguins fill the upper bins. Equal-width bins follow the range of the values; quantile bins follow how many values there are.

</details>

---

## 4. Feature Scaling: Bringing Features onto a Similar Range
Feature scaling transforms numerical features so that they share a similar scale.

For example, consider these two features:
+ `income` ranges from 20_000 to 100_000 euro
+ `age` ranges from 20 to 100 years

Without scaling, `income` would dominate [distance-based algorithms](https://towardsdatascience.com/all-about-feature-scaling-bcc0ad75cb35/) because its range of values is much larger than that of `age`.

Feature scaling adjusts `income` and `age` so that they **contribute equally to what the model learns**, which can improve the model.


### 4.1 Min-Max Scaling

Min-max scaling transforms all values of a numerical feature to a fixed range, typically 0 to 1, by subtracting the minimum value and dividing by the range:

$$\large X_{scaled} = \large \dfrac{X - X_{min}}{X_{max} - X_{min}}$$

After transformation:
+ Features are constrained to a fixed range (e.g., 0 to 1)
  + The minimum value becomes 0 and the maximum value becomes 1
+ The relative distances between data points are preserved
  + Values in between 0 and 1 keep their proportional positions
+ Outliers are not handled well: a single extreme value compresses all other values into a narrow range.

Documentation: [`MinMaxScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html).

In [ ]:
from sklearn.preprocessing import MinMaxScaler

In [ ]:
# Instantiate the scaler
min_max_scaler = MinMaxScaler(feature_range=(0, 1))

In [ ]:
# Fit the scaler to the columns 'bill_length_mm' and 'flipper_length_mm' of the training data
min_max_scaler.fit(X=X_train[["bill_length_mm", "flipper_length_mm"]])

In [ ]:
# Learnt Maximum values for each feature on the train data
print("Max values:", min_max_scaler.data_max_)

# Learnt Minimum values for each feature on the train data
print("Min values:", min_max_scaler.data_min_)

# Learnt Range for each feature on the train data
print("Range:", min_max_scaler.data_range_)

In [ ]:
X_train[["bill_length_mm", "flipper_length_mm"]].max()

In [ ]:
# Scale the features
min_max_scaled_bill_fli_df_train = min_max_scaler.transform(
    X=X_train[["bill_length_mm", "flipper_length_mm"]]
)
min_max_scaled_bill_fli_df_train

In [ ]:
min_max_scaled_bill_fli_df_train.describe()

**Note** that the **min** and **max** values of each feature are 0 and 1 respectively.

The test data is scaled with the minimum and maximum learned from the **training** data. Its values can therefore fall slightly outside the range 0 to 1, which is expected: the test set must not influence the scaler.

In [ ]:
min_max_scaled_bill_fli_df_test = min_max_scaler.transform(
    X=X_test[["bill_length_mm", "flipper_length_mm"]]
)
min_max_scaled_bill_fli_df_test.describe().loc[["min", "max"]]

In [ ]:
fig, ax = plt.subplots(nrows=2, ncols=2)

# Plotting unscaled data
sns.histplot(data=X_train, x="bill_length_mm", ax=ax[0, 0], kde=True)
ax[0, 0].set_title("Unscaled Bill Length")

sns.histplot(data=X_train, x="flipper_length_mm", ax=ax[0, 1], kde=True)
ax[0, 1].set_title("Unscaled Flipper Length")

# Plotting scaled data
sns.histplot(
    data=min_max_scaled_bill_fli_df_train, x="bill_length_mm", ax=ax[1, 0], kde=True
)
ax[1, 0].set_title("Scaled Bill Length")

sns.histplot(
    data=min_max_scaled_bill_fli_df_train, x="flipper_length_mm", ax=ax[1, 1], kde=True
)
ax[1, 1].set_title("Scaled Flipper Length")

# Adjust layout
plt.tight_layout()

As the plots above show, the shape of each distribution is preserved after min-max scaling: only the axis values change.

### 4.2 Standard Scaling

Standard scaling transforms the values of a numerical feature so that they have a mean of zero and a standard deviation of one. It subtracts the mean of the feature from each value and then divides by the standard deviation:

$$\large X_{\text{standardized}} = \frac{X - \mu}{\sigma}$$

Where:
- $\mu$ is the mean of the feature values.
- $\sigma$ is the standard deviation of the feature values.

**After transformation:**
+ **Centred data**: the mean of the transformed data is 0.
+ **Unit variance**: the standard deviation becomes 1, so the variance of every feature is the same.
+ **Preserved relationships**: the relative distances between data points are maintained, as with min-max scaling.
+ **Outliers**: the result has no fixed range, so a single outlier does not squeeze all other values into a narrow interval as with min-max scaling. The mean and standard deviation are still influenced by outliers, though.

You already used [`StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) in W3D2 (notebook 02 of the regularisation repository), before fitting Ridge and Lasso. It is the most common choice for models that penalise or compare coefficients.

### 4.3 Robust Scaling
Robust scaling transforms the values of a numerical feature by subtracting the median and then dividing by the interquartile range (IQR):

$$\large X_{\text{robust}} = \frac{X - \text{median}(X)}{\text{IQR}}$$

Where:
- **median(X)** is the median of the feature values.
- **IQR** is the interquartile range: the difference between the 75th percentile (Q3) and the 25th percentile (Q1) of the feature values.

After transformation:
+ **Centred and scaled data**: the median of the transformed data becomes 0, and dividing by the IQR puts the features on a comparable scale.
+ **Outliers**: the median and IQR are less affected by outliers than the mean and standard deviation, so this method is better suited to datasets with outliers.
+ **Preserved relationships**: like the other scaling methods, robust scaling keeps the relative distances between data points.


In scikit-learn: [`RobustScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.RobustScaler.html).

### 4.4 Log Scaling
Log scaling is a non-linear transformation that uses the logarithm to reduce the range and variation of data values.

A **log transformation** applies a logarithm to the values of a numerical feature, typically the natural logarithm (base e), although any base can be used depending on the data. The transformation is:

$$\large X_{\text{log}} = \log(X)$$

Where:
- **log()** denotes the logarithm, to any base.

**After transformation:**
+ **Smaller differences in scale**: large values are compressed much more than small values.
+ **Less skewness**: a right-skewed distribution becomes more symmetric.
+ **Stabilised variance**: the spread around large values shrinks more than the spread around small values, which makes the variance more uniform across the dataset.
+ **Smaller impact of outliers**: outliers far from most data points have less influence after the transformation, because high values are compressed the most.

**Considerations:**
- The logarithm is only defined for positive values. For data that contains zeros, the usual choice is `np.log1p`, which computes $\log(x + 1)$.


scikit-learn has no dedicated log transformer, but [`FunctionTransformer`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.FunctionTransformer.html) turns any function into one, with the usual `.fit()` and `.transform()` methods. Here we wrap `np.log1p`.

In [ ]:
from sklearn.preprocessing import FunctionTransformer

In [ ]:
# Create the transformer from np.log1p, which computes log(x + 1)
log_transformer = FunctionTransformer(func=np.log1p)

In [ ]:
# Fit the transformer to the 'flipper_length_mm' column of X_train
log_transformer.fit(X=X_train[["flipper_length_mm"]])

In [ ]:
# Transform the data using the fitted transformer
log_transformed_flip_df_train = log_transformer.transform(
    X_train[["flipper_length_mm"]]
)
log_transformed_flip_df_train

In [ ]:
fig, ax = plt.subplots(nrows=2, ncols=1)

# Plotting unscaled data
sns.histplot(data=X_train, x="flipper_length_mm", ax=ax[0], kde=True)
ax[0].set_title("Original Flipper Length")

sns.histplot(
    data=log_transformed_flip_df_train, x="flipper_length_mm", ax=ax[1], kde=True
)
ax[1].set_title("Log-Transformed Flipper Length")

# Adjust layout
plt.tight_layout()

### Further Reading on Scaling
+ [Compare the effect of different scalers on data with outliers](https://scikit-learn.org/stable/auto_examples/preprocessing/plot_all_scaling.html#sphx-glr-auto-examples-preprocessing-plot-all-scaling-py)
+ [Importance of feature scaling](https://scikit-learn.org/stable/auto_examples/preprocessing/plot_scaling_importance.html)


---

## 5. Feature Expansion
### `PolynomialFeatures`
You used `PolynomialFeatures` in W3D2: with `degree` to fit curves (notebook 01 of the regularisation repository) and with `interaction_only=True` to create interaction terms (notebook 02). Here we look at what the new columns mean.

We can use the scikit-learn [`PolynomialFeatures()`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html) to add polynomial or interaction features to a dataset. This lets a linear model capture non-linear relationships and interactions between features. The key parameters are:

* `degree`: the degree of the polynomial features (default 2). For example, for a single feature $X$, `degree=2` generates $X, X^2$.

* `interaction_only`: if `True`, produces only products of distinct input features. For example, for two features $X_1$ and $X_2$, it generates $X_1 \times X_2$ but not $X_1^2$ or $X_2^2$.

* `include_bias`: decides whether to include a bias column (a column of ones). Set it to `False` if the model already handles the intercept.

> [!IMPORTANT]
> Scale your features first, then apply the polynomial feature expansion.

### 5.1 Polynomial Terms

- Polynomial terms are additional features obtained by raising an existing feature to a power.
- They allow a model to capture non-linear relationships.
- For a feature x, consider the model:

$$
y = a_0 + a_1x + a_2x^2 +\ldots+\epsilon
$$

- They can improve model accuracy, but they increase the risk of overfitting.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

In [ ]:
# Instantiate the polynomial features object
poly_expansion = PolynomialFeatures(
    degree=2, interaction_only=False, include_bias=False
)

In [ ]:
poly_expansion.fit(min_max_scaled_bill_fli_df_train[["bill_length_mm"]])

In [ ]:
expanded_scaled_bill_length_df_train = poly_expansion.transform(
    min_max_scaled_bill_fli_df_train[["bill_length_mm"]]
)
expanded_scaled_bill_length_df_train

### 5.2 Interaction Terms

- With several features, there can also be *interactions* (products of different features).
- For 2 features, $x_0$ and $x_1$ for example, a 2nd-degree polynomial may contain:

$$
1,~x_0,~x_1,~x_0^2,~x_0x_1,~x_1^2
$$

- Each term gets its own coefficient in a regression model.
- Set `interaction_only=True` to keep only the interaction terms and drop the squared terms.

In [ ]:
# Instantiate the polynomial features object
poly_expansion = PolynomialFeatures(
    degree=2, interaction_only=False, include_bias=False
)

`PolynomialFeatures` does not support NaNs, so we must handle the missing values before calling `.fit()`.

Let's check for missing values first.

In [ ]:
min_max_scaled_bill_fli_df_train.isna().sum()

There are two missing values in the `flipper_length_mm` column. We could apply any of the strategies from earlier, such as imputing or dropping. Because only a little data is lost, we simply drop these rows.

In [ ]:
# drop missings in data frame
min_max_scaled_bill_fli_df_train.dropna(inplace=True)

In [ ]:
# check for missings
min_max_scaled_bill_fli_df_train.isna().sum()

In [ ]:
poly_expansion.fit(X=min_max_scaled_bill_fli_df_train)

In [ ]:
expanded_scaled_bill_fli_length_df_train = poly_expansion.transform(
    min_max_scaled_bill_fli_df_train
)
expanded_scaled_bill_fli_length_df_train

**Note** that the column `bill_length_mm flipper_length_mm` is the interaction term. With `interaction_only=True`, the quadratic features `bill_length_mm^2` and `flipper_length_mm^2` would not be present.

### 5.3 Creating Features by Hand

`PolynomialFeatures` creates new columns mechanically: every square and every product, whether or not it means anything. Knowledge about the data often suggests a better feature.

A penguin's bill can be long and thin or short and deep. Neither `bill_length_mm` nor `bill_depth_mm` describes that **shape** on its own, but their ratio does:

$$
\text{bill ratio} = \frac{\text{bill length}}{\text{bill depth}}
$$

Creating this feature does not leak information from the test set: each penguin's ratio comes from its own two measurements, and nothing is learned from the other penguins. That is why it can be calculated on the training and the test set directly, without fitting anything. A missing length or depth gives a missing ratio, which an imputer can fill later.

In [ ]:
bill_ratio_train = X_train["bill_length_mm"] / X_train["bill_depth_mm"]
bill_ratio_test = X_test["bill_length_mm"] / X_test["bill_depth_mm"]

bill_ratio_train.head()

Does the new feature tell us more about body mass than its two parts? Compare their correlations with the target on the training set.

In [ ]:
pd.Series(
    {
        "bill_length_mm": X_train["bill_length_mm"].corr(y_train),
        "bill_depth_mm": X_train["bill_depth_mm"].corr(y_train),
        "bill_ratio": bill_ratio_train.corr(y_train),
        "flipper_length_mm": X_train["flipper_length_mm"].corr(y_train),
    }
).round(3)

The ratio correlates more strongly with body mass (0.694) than bill length (0.565) or bill depth (-0.485) on their own: combining the two measurements adds information. It does not replace every other feature, though: `flipper_length_mm` is still the strongest single predictor (0.859).

The ratio also describes the species. Compare its mean and standard deviation for each species in the training set.

In [ ]:
bill_ratio_train.groupby(X_train["species"]).agg(["mean", "std"]).round(2)

Gentoo penguins have long, thin bills (a mean ratio of 3.18), Adelie penguins short, deep ones (2.11), and Chinstrap penguins lie in between (2.66). Within each species the ratio varies little (a standard deviation of 0.15 to 0.17), so one number captures much of what distinguishes the three bill shapes.

### Try It: The Bill Ratio as a Transformer

To use a hand-made feature inside a pipeline (notebook 03), wrap it in a `FunctionTransformer`, as you did for the log transformation in section 4.4. Complete the function so that it returns a DataFrame with one column, `bill_ratio`, then apply the transformer to the training and the test set. Check that the result matches `bill_ratio_test` from above.

In [ ]:
def make_bill_ratio(X):
    """Return a DataFrame with the bill ratio of each penguin."""
    # TODO: divide bill_length_mm by bill_depth_mm
    return pd.DataFrame({"bill_ratio": ...}, index=X.index)


# TODO: wrap make_bill_ratio in a FunctionTransformer
ratio_transformer = ...

bill_columns = ["bill_length_mm", "bill_depth_mm"]
ratio_train = ratio_transformer.fit_transform(X_train[bill_columns])

# TODO: transform the test set with the same transformer
ratio_test = ...

ratio_test["bill_ratio"].equals(bill_ratio_test)

<details><summary>Show answer</summary>

```python
def make_bill_ratio(X):
    """Return a DataFrame with the bill ratio of each penguin."""
    return pd.DataFrame({"bill_ratio": X["bill_length_mm"] / X["bill_depth_mm"]}, index=X.index)


ratio_transformer = FunctionTransformer(func=make_bill_ratio)

bill_columns = ["bill_length_mm", "bill_depth_mm"]
ratio_train = ratio_transformer.fit_transform(X_train[bill_columns])
ratio_test = ratio_transformer.transform(X_test[bill_columns])

ratio_test["bill_ratio"].equals(bill_ratio_test)
```

The check prints `True`: the transformer calculates exactly the same ratio. Fitting it learns nothing, because the ratio of each penguin depends only on its own row. What the transformer adds is the scikit-learn interface, so the new feature can sit inside a `ColumnTransformer` next to the imputers, encoders and scalers.

</details>

---

## Bonus: Smarter Imputation

Earlier, we used basic imputation methods that rely on simple statistics, such as the **mean** or **median** of the column with missing values. More advanced imputation techniques also take the relationships between different features into account.

### Group-Wise Mean, Median or Mode Imputation
This method fills in missing values with the mean, median or mode calculated within subgroups of the data.

To build a group-wise imputer with the scikit-learn `.fit()` and `.transform()` interface, we create a custom class based on scikit-learn's `BaseEstimator` and `TransformerMixin` classes.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

In [ ]:
class GroupMeanImputer(BaseEstimator, TransformerMixin):
    def __init__(self, group_column, target_columns):
        # Initialize with the name of the group column and a list of target columns
        self.group_column = group_column
        self.target_columns = target_columns

    def fit(self, X, y=None):
        # Calculate the mean for each target column within each group
        self.means_ = {}
        # Calculate the unique categories in the group  column
        self.categories_ = X[self.group_column].unique()
        for column in self.target_columns:
            self.means_[column] = X.groupby(self.group_column)[column].mean().to_dict()
        return self

    def transform(self, X):
        # Apply the learned means_ to fill in missing values for each target column
        X = X.copy()
        for column in self.target_columns:
            # Get the group means_ for the current column
            for category in self.categories_:
                group_means = self.means_[column][category]
                # Fill missing values for this category's rows based on its group mean
                mask = (X[self.group_column] == category) & X[column].isna()
                X.loc[mask, column] = group_means

        return X[self.target_columns]

In [ ]:
# Instantiate the group-wise imputer
group_mean_imputer = GroupMeanImputer(
    group_column="island", target_columns=["bill_depth_mm", "flipper_length_mm"]
)

In [ ]:
# Fit the imputer
group_mean_imputer.fit(X=X_train)

In [ ]:
group_mean_imputer.means_

In [ ]:
group_mean_imputer.categories_

In [ ]:
group_mean_imputer.transform(X=X_train)

### Model Imputation
+ Use a model to predict the missing values from the other variables.
+ For iterative imputation, scikit-learn provides [IterativeImputer()](https://scikit-learn.org/stable/auto_examples/impute/plot_iterative_imputer_variants_comparison.html#sphx-glr-auto-examples-impute-plot-iterative-imputer-variants-comparison-py).

### K-Nearest Neighbours (KNN) Imputation
+ Fills in missing entries using the k nearest neighbours of each point, found by measuring the distance to the other points.
+ scikit-learn provides [KNNImputer()](https://scikit-learn.org/stable/modules/generated/sklearn.impute.KNNImputer.html).

## Summary

In this notebook you:

- Imputed missing values with `SimpleImputer`, fitting it on the training data and only transforming the test data.
- Encoded the nominal `species` feature with `OneHotEncoder` and an ordinal education level with `OrdinalEncoder`.
- Discretised `bill_length_mm` into five quantile bins with `KBinsDiscretizer`.
- Applied min-max scaling and a log transformation, and saw that the test data is scaled with what was learned on the training data.
- Created a bill ratio feature by hand, compared its correlation with body mass with those of its two parts, and wrapped it in a `FunctionTransformer`.

You also revisited polynomial and interaction features from W3D2, and saw that every transformer follows the same pattern: `.fit()` on the training data, then `.transform()` on any data. Notebook 03 uses that pattern to chain all of these steps into one pipeline.

## References & Further Reading

- [**Preprocessing data (scikit-learn)**](https://scikit-learn.org/stable/modules/preprocessing.html): the user guide for encoding, discretisation and scaling.
- [**Imputation of missing values (scikit-learn)**](https://scikit-learn.org/stable/modules/impute.html): `SimpleImputer`, `KNNImputer` and `IterativeImputer`.
- [**Compare the effect of different scalers on data with outliers (scikit-learn)**](https://scikit-learn.org/stable/auto_examples/preprocessing/plot_all_scaling.html): a visual comparison of the scalers in section 4.